# Retrieval Augmented Generation (RAG) in LangChain

## 1. What is RAG?

**RAG (Retrieval Augmented Generation)** is an architecture that improves the responses of an LLM by giving it relevant external information retrieved from a knowledge source before generating the final answer.

Instead of asking an LLM to answer only from its trained knowledge:

    User Query
         ↓
        LLM
         ↓
      Answer

RAG works like:

    User Query
         ↓
      Retriever
         ↓
   Relevant Documents
         ↓
      LLM + Context
         ↓
      Final Answer

### Simple Definition

> RAG = Retrieve relevant information + Augment the prompt + Generate an answer.

---

# 2. Why Do We Need RAG?

LLMs have several limitations:

1. Their knowledge can become outdated.
2. They may not know private/company-specific information.
3. They can hallucinate.
4. Retraining an LLM for every new document is expensive.
5. LLMs cannot automatically access every external knowledge source.
6. Large collections of documents cannot simply be placed entirely inside every prompt.

RAG solves many of these problems by retrieving relevant information at query time.

### Example

Suppose a company has:

    employee_handbook.pdf
    company_policies.pdf
    leave_policy.pdf
    salary_policy.pdf

An LLM may not know the contents of these documents.

With RAG:

    User:
    "How many annual leaves can an employee take?"

    ↓

    Retriever searches company documents

    ↓

    Relevant section from leave_policy.pdf

    ↓

    LLM receives the retrieved context

    ↓

    "According to the company leave policy, employees
     are entitled to ..."

The LLM is not expected to memorize the company documents.

---

# 3. RAG Full Form

RAG stands for:

    Retrieval
    Augmented
    Generation

### Retrieval

Find relevant information from an external knowledge source.

### Augmented

Add the retrieved information to the LLM's context/prompt.

### Generation

The LLM generates an answer using the retrieved context.

---

# 4. Basic RAG Architecture

A typical RAG system has two major phases:

1. Indexing / Ingestion
2. Retrieval + Generation

Architecture:

    ┌──────────────────────────┐
    │      Data Sources       │
    │ PDF / Web / DB / Docs   │
    └────────────┬─────────────┘
                 ↓
          Document Loader
                 ↓
            Text Splitter
                 ↓
             Embeddings
                 ↓
           Vector Store
                 │
                 │
          ───────┼────────
                 │
             User Query
                 ↓
             Retriever
                 ↓
        Relevant Documents
                 ↓
        Prompt + Context
                 ↓
                LLM
                 ↓
          Generated Answer

---

# 5. RAG Pipeline

The complete RAG pipeline can be represented as:

    Documents
        ↓
    Load Documents
        ↓
    Split Documents
        ↓
    Create Embeddings
        ↓
    Store Vectors
        ↓
    User Query
        ↓
    Embed Query
        ↓
    Retrieve Relevant Chunks
        ↓
    Optional Filtering / Reranking
        ↓
    Build Prompt
        ↓
    Send Context + Query to LLM
        ↓
    Generate Answer

---

# 6. Two Phases of RAG

## Phase 1: Indexing / Ingestion

This happens before the user asks questions.

    Documents
       ↓
    Load
       ↓
    Split
       ↓
    Embed
       ↓
    Store

The purpose is to prepare the knowledge base for efficient retrieval.

---

## Phase 2: Query / Retrieval

This happens when the user asks a question.

    User Query
       ↓
    Retrieve
       ↓
    Relevant Context
       ↓
    Prompt
       ↓
    LLM
       ↓
    Answer

---

# 7. RAG Indexing Pipeline

## Step 1: Data Sources

RAG can work with many types of data:

- PDF
- TXT
- CSV
- JSON
- Markdown
- HTML
- Word documents
- Websites
- Databases
- APIs
- Cloud storage
- Company knowledge bases

Example:

    company_policy.pdf

---

# 8. Step 2: Document Loading

LangChain uses **Document Loaders** to load external data.

Example:

    from langchain_community.document_loaders import PyPDFLoader

    loader = PyPDFLoader("company_policy.pdf")
    documents = loader.load()

The output is generally a list of `Document` objects.

A `Document` contains:

    page_content
    metadata

Example:

    Document(
        page_content="Employees are entitled to...",
        metadata={
            "source": "company_policy.pdf",
            "page": 5
        }
    )

---

# 9. Step 3: Text Splitting

Large documents are divided into smaller chunks.

Why?

Because sending an entire large document to an LLM is inefficient and can exceed the context window.

Example:

    Large Document
          ↓
    ┌─────────────┐
    │ Chunk 1     │
    ├─────────────┤
    │ Chunk 2     │
    ├─────────────┤
    │ Chunk 3     │
    ├─────────────┤
    │ Chunk 4     │
    └─────────────┘

Common LangChain splitters include:

- CharacterTextSplitter
- RecursiveCharacterTextSplitter
- TokenTextSplitter
- MarkdownHeaderTextSplitter
- HTMLHeaderTextSplitter
- RecursiveJsonSplitter
- Language-specific splitters

Example:

    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

---

# 10. Chunk Size

`chunk_size` determines approximately how large each chunk should be.

Example:

    chunk_size = 1000

means the splitter attempts to create chunks around that size according to its length function and splitting strategy.

Too small:

    Document
      ↓
    Tiny chunks

Problems:

- Loss of context
- Poor retrieval quality
- More chunks

Too large:

    Document
      ↓
    Huge chunks

Problems:

- More irrelevant information
- Higher token usage
- Less precise retrieval

---

# 11. Chunk Overlap

`chunk_overlap` keeps some content from the previous chunk in the next chunk.

Example:

    Chunk 1:
    A B C D E F

    Chunk 2:
            E F G H I J

Here:

    E F

is the overlap.

Why?

It prevents important information from being lost at chunk boundaries.

---

# 12. Step 4: Embeddings

After splitting the documents, each chunk is converted into a vector using an embedding model.

Example:

    "Python is a programming language."

becomes something conceptually like:

    [0.12, -0.34, 0.87, 0.21, ...]

This vector represents the semantic meaning of the text.

Example:

    Chunk 1 → [0.12, 0.44, 0.91, ...]
    Chunk 2 → [0.72, 0.11, 0.32, ...]
    Chunk 3 → [0.15, 0.83, 0.62, ...]

---

# 13. Why Embeddings Are Important

Embeddings allow the system to compare semantic meaning.

For example:

    Query:
    "How much vacation can I take?"

    Document:
    "Employees receive 20 days of annual leave."

Although the words are different:

    vacation
    annual leave

their semantic meanings are related.

Embeddings help the retriever identify this relationship.

---

# 14. Step 5: Vector Store

The embeddings and associated document information are stored in a vector store.

Conceptually:

    ┌───────────────┐
    │ Vector        │
    │ Embedding     │
    ├───────────────┤
    │ Text          │
    ├───────────────┤
    │ Metadata      │
    └───────────────┘

Examples of vector stores/databases:

- FAISS
- Chroma
- Pinecone
- Qdrant
- Weaviate
- Milvus
- PGVector
- Elasticsearch
- Redis
- MongoDB Atlas Vector Search

---

# 15. Vector Store vs Vector Database

These terms are sometimes used interchangeably, but they are not always identical.

### Vector Store

Primarily focused on storing and searching vectors.

Example:

    FAISS

### Vector Database

A production database designed to manage vector data along with operational features such as:

- persistence
- metadata filtering
- scaling
- distributed search
- indexing
- updates
- access control

Examples:

    Pinecone
    Qdrant
    Weaviate
    Milvus

---

# 16. Creating a Vector Store

Conceptually:

    Documents
        ↓
    Embedding Model
        ↓
    Vector Store

Example:

    from langchain_chroma import Chroma

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings
    )

The vector store now contains searchable document representations.

---

# 17. Query Phase

Now suppose the user asks:

    "What is the annual leave policy?"

The query is processed.

    User Query
        ↓
    Embedding Model
        ↓
    Query Vector
        ↓
    Vector Store
        ↓
    Similar Documents
        ↓
    Retriever

---

# 18. Query Embedding

The query is converted into an embedding using the same compatible embedding model used for the indexed documents.

Example:

    Query:
    "What is the annual leave policy?"

becomes:

    [0.14, 0.42, 0.89, ...]

The vector store compares this query vector against stored document vectors.

---

# 19. Similarity Search

The system calculates similarity between:

    Query Vector

and

    Document Vectors

Common similarity/distance metrics include:

- Cosine similarity
- Euclidean distance
- Dot product

The most relevant chunks are returned.

Example:

    Query
      ↓
    Vector Search
      ↓
    Chunk 5   ← highly relevant
    Chunk 12  ← relevant
    Chunk 8   ← relevant
    Chunk 21  ← less relevant

---

# 20. Top-K Retrieval

`k` determines how many documents/chunks are returned.

Example:

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 4}
    )

This requests approximately the top 4 relevant documents/chunks.

Conceptually:

    Query
      ↓
    Search
      ↓
    Top 4 chunks
      ↓
    LLM

---

# 21. Retriever

A retriever provides a standard interface for retrieving relevant documents for a query.

Input:

    Query

Output:

    List[Document]

Example:

    retriever = vectorstore.as_retriever()

    docs = retriever.invoke(
        "What is the annual leave policy?"
    )

The retriever is responsible for finding relevant context.

It does NOT normally generate the final answer.

---

# 22. Vector Store vs Retriever

### Vector Store

Responsible for:

- storing vectors
- similarity search
- storing document information
- metadata filtering

### Retriever

Responsible for:

- accepting a query
- retrieving relevant documents
- providing a consistent retrieval interface

Relationship:

    Vector Store
         ↓
    as_retriever()
         ↓
      Retriever
         ↓
    Relevant Documents

---

# 23. Retrieval Strategies

RAG does not require only basic similarity search.

Common strategies include:

### 1. Similarity Search

Returns the most semantically similar chunks.

### 2. MMR

Maximum Marginal Relevance balances:

    Relevance + Diversity

Useful when the top results are too similar to each other.

### 3. Similarity Score Threshold

Returns documents only if their similarity score passes a threshold.

### 4. Multi-Query Retrieval

Uses an LLM to generate multiple variations of the user's query.

### 5. Hybrid Retrieval

Combines semantic/vector search with keyword-based search.

### 6. Reranking

Retrieves candidates first and then reorders them using a stronger ranking model.

---

# 24. MMR in RAG

Suppose the query is:

    "What are the benefits of remote work?"

Normal similarity search may return:

    Chunk A → remote work benefits
    Chunk B → remote work benefits
    Chunk C → remote work benefits

These may be very similar.

MMR tries to return:

    Chunk A → productivity
    Chunk B → employee satisfaction
    Chunk C → cost savings

So the final context can contain more diverse information.

---

# 25. Metadata

Documents can contain metadata such as:

    source
    page
    author
    department
    date
    document_type
    access_level

Example:

    metadata = {
        "source": "leave_policy.pdf",
        "department": "HR",
        "year": 2026
    }

Metadata can be used for filtering.

Example:

    Query:
    "What is the HR leave policy?"

    Filter:
    department = "HR"

This can reduce irrelevant retrieval.

---

# 26. Prompt Augmentation

After retrieval, the retrieved documents are inserted into the prompt sent to the LLM.

Conceptually:

    System Instructions

    + Retrieved Context

    + User Question

    ↓

    LLM

Example prompt structure:

    Use the following context to answer the question.

    Context:
    {retrieved_documents}

    Question:
    {user_question}

    Answer:

The retrieved documents are called the **context**.

---

# 27. Generation

The LLM receives:

    User Question
          +
    Retrieved Context
          +
    Instructions

and generates the final response.

Example:

    Question:
    "How many annual leaves are available?"

    Retrieved Context:
    "Employees are entitled to 20 annual leave days."

    LLM:

    "According to the provided company policy,
     employees are entitled to 20 annual leave days."

---

# 28. Grounded Generation

A major goal of RAG is **grounding**.

Grounding means generating the answer based on retrieved evidence rather than relying only on the model's internal knowledge.

Conceptually:

    Knowledge Base
          ↓
       Evidence
          ↓
          LLM
          ↓
    Grounded Answer

A good RAG prompt can instruct the model:

    Answer using only the provided context.
    If the answer is not present in the context,
    say that the information is unavailable.

This can reduce unsupported answers.

---

# 29. Hallucination in RAG

RAG does NOT completely eliminate hallucinations.

Possible causes:

- Wrong documents retrieved
- Insufficient context
- Poor chunking
- Ambiguous query
- Poor prompt
- LLM ignores context
- Conflicting documents
- Outdated data

Therefore:

    RAG ≠ Zero Hallucination

A better mental model is:

    Good Retrieval
         +
    Good Context
         +
    Good Prompt
         +
    Good LLM
         ↓
    Better Grounded Answers

---

# 30. RAG in LangChain

LangChain can connect:

    Document Loaders
         ↓
    Text Splitters
         ↓
    Embeddings
         ↓
    Vector Stores
         ↓
    Retrievers
         ↓
    Prompt Templates
         ↓
    Chat Models
         ↓
    RAG Chain

LangChain provides abstractions so these components can be composed together.

---

# 31. Simple LangChain RAG Architecture

Conceptually:

    loader
       ↓
    splitter
       ↓
    embeddings
       ↓
    vectorstore
       ↓
    retriever
       ↓
    prompt
       ↓
    LLM
       ↓
    answer

---

# 32. Simple RAG Example

A simplified LangChain RAG implementation may look like:

    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_chroma import Chroma

    # Load documents
    documents = loader.load()

    # Split documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

    # Create vector store
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    # Create retriever
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 4}
    )

    # Retrieve documents
    docs = retriever.invoke(
        "What is the leave policy?"
    )

The exact imports and model configuration depend on the integrations being used.

---

# 33. RAG with LCEL

LangChain Expression Language (LCEL) can be used to compose a RAG pipeline.

Conceptually:

    Question
       ↓
    Retriever
       ↓
    Context
       ↓
    Prompt
       ↓
    LLM
       ↓
    Parser
       ↓
    Answer

Example pattern:

    rag_chain = (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | parser
    )

Then:

    answer = rag_chain.invoke(
        "What is the leave policy?"
    )

---

# 34. Why RunnablePassthrough Is Useful

Suppose the chain needs two values:

    context
    question

The retriever generates:

    context

But the original question must also be passed to the prompt.

Conceptually:

    User Question
        ├──────────────→ Retriever → Context
        │
        └──────────────→ Passthrough → Question

Then:

    Context + Question
            ↓
          Prompt
            ↓
           LLM

---

# 35. RAG Prompt

A typical RAG prompt can contain:

    System:
    You are a helpful assistant.
    Answer using only the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:

The prompt should clearly separate:

- Instructions
- Context
- User question

---

# 36. Document Formatting

Retrieved documents often need to be converted into a clean string before being inserted into a prompt.

Conceptually:

    List[Document]
          ↓
    Format Documents
          ↓
    Context String
          ↓
    Prompt

Example:

    def format_docs(docs):
        return "\n\n".join(
            doc.page_content
            for doc in docs
        )

This makes the retrieved content easier for the LLM to consume.

---

# 37. Basic RAG Chain Flow

    User Question
          ↓
       Retriever
          ↓
    List[Documents]
          ↓
    format_docs()
          ↓
       Context
          ↓
       Prompt
          ↓
         LLM
          ↓
       Parser
          ↓
       Answer

---

# 38. RAG vs Fine-Tuning

These are different techniques.

## RAG

Adds external information at inference time.

    Query
      ↓
    Retrieve Knowledge
      ↓
    LLM
      ↓
    Answer

Useful for:

- Frequently changing information
- Private documents
- Knowledge bases
- Company data
- Documentation
- Search-based applications

---

## Fine-Tuning

Changes model behavior by training it on additional examples/data.

    Training Data
         ↓
    Fine-Tuning
         ↓
    Updated Model
         ↓
    Query
         ↓
    Answer

Useful for:

- Behavior
- Style
- Specialized task performance
- Output formatting
- Domain/task adaptation

---

# 39. RAG vs Fine-Tuning Comparison

| Feature | RAG | Fine-Tuning |
|---|---|---|
| Adds external knowledge | Yes | Not directly |
| Uses documents at query time | Yes | No |
| Good for private knowledge | Yes | Possible, but less direct |
| Good for frequently changing data | Yes | No |
| Changes model behavior | Limited | Yes |
| Requires model training | No | Yes |
| Easy to update knowledge | Usually yes | Requires retraining/update process |
| Main focus | Knowledge retrieval | Model behavior/task adaptation |

Important:

> RAG and fine-tuning are not mutually exclusive.

A system can use both.

---

# 40. RAG vs Long Context

Modern LLMs may support very large context windows.

This does not mean RAG is unnecessary.

Without retrieval:

    Huge Knowledge Base
           ↓
    Send large amount of text
           ↓
          LLM

Problems:

- High token usage
- Higher latency
- More noise
- Potential context dilution
- Higher cost

With RAG:

    Huge Knowledge Base
           ↓
       Retrieve
           ↓
    Relevant Information
           ↓
          LLM

RAG reduces the amount of information that must be provided to the model.

---

# 41. Naive RAG

A simple RAG architecture is:

    Documents
       ↓
    Chunking
       ↓
    Embeddings
       ↓
    Vector Store
       ↓
    Similarity Search
       ↓
    Prompt
       ↓
    LLM

This is often called **Naive RAG** because it uses a relatively straightforward retrieval pipeline.

---

# 42. Advanced RAG

Advanced RAG may include:

    Query
      ↓
    Query Transformation
      ↓
    Hybrid Retrieval
      ↓
    Metadata Filtering
      ↓
    Candidate Retrieval
      ↓
    Reranking
      ↓
    Context Compression
      ↓
    Prompt Construction
      ↓
    LLM
      ↓
    Answer + Citations

---

# 43. Query Transformation

Sometimes the original user query is not optimal for retrieval.

Example:

    User:
    "What happens if I don't use all my leaves?"

A system may transform it into:

    "Does unused annual leave carry forward?"

This can improve retrieval.

Techniques include:

- Query rewriting
- Query expansion
- Multi-query retrieval
- Step-back prompting
- HyDE

---

# 44. Multi-Query Retrieval

A single question can be converted into multiple search queries.

Example:

    Original:
    "How does remote work affect productivity?"

Generated queries:

    "remote work productivity"
    "work from home employee performance"
    "remote work efficiency"
    "productivity benefits of remote work"

Each query retrieves documents.

The results are combined.

This can improve recall.

---

# 45. HyDE

**HyDE = Hypothetical Document Embeddings**

Instead of directly embedding the user's question:

    Query
      ↓
    LLM generates hypothetical answer/document
      ↓
    Embed hypothetical document
      ↓
    Retrieve similar documents

The hypothetical answer is used as a retrieval aid.

Important:

> HyDE does not mean the hypothetical answer is treated as factual evidence.

The actual documents still need to be retrieved and used as evidence.

---

# 46. Hybrid Search

Hybrid search combines different retrieval methods.

For example:

    Semantic Search
          +
    Keyword Search
          ↓
       Combined
          ↓
       Results

Semantic search is useful for meaning.

Keyword search is useful for:

- exact names
- product IDs
- error codes
- technical terms
- legal clauses

Hybrid retrieval can therefore outperform a single retrieval strategy for many applications.

---

# 47. Reranking

Initial retrieval may return:

    20 candidate chunks

A reranker evaluates these candidates more carefully.

    Query
      ↓
    Retriever
      ↓
    20 candidates
      ↓
    Reranker
      ↓
    Top 5
      ↓
    LLM

This is often called a two-stage retrieval architecture.

### Stage 1

High recall.

Retrieve many potentially relevant documents.

### Stage 2

High precision.

Rerank and select the best documents.

---

# 48. Contextual Compression

Retrieved documents may contain unnecessary information.

Contextual compression attempts to extract only the relevant portions.

Example:

    Retrieved Document
          ↓
    Relevant Information
          ↓
        LLM

Instead of passing an entire large document to the LLM.

Benefits:

- Lower token usage
- Less noise
- Better context quality

---

# 49. Parent Document Retrieval

Small chunks can improve retrieval precision, but very small chunks may lose context.

Parent Document Retriever addresses this tradeoff.

Conceptually:

    Parent Document
          ↓
    Small Child Chunks
          ↓
    Retrieve Child Chunk
          ↓
    Return Parent Context

This allows:

    Small chunks → precise retrieval
    Parent document → richer context

---

# 50. Self-Query Retriever

A Self-Query Retriever uses an LLM to convert a natural-language query into:

    Semantic Query
        +
    Metadata Filter

Example:

    User:
    "Show me documents about Python published after 2025."

The system may interpret:

    Semantic query:
    "Python"

    Metadata filter:
    year > 2025

This is useful when metadata is important.

---

# 51. Multi-Vector Retrieval

One document can have multiple representations.

For example:

    Document
       ├── Summary vector
       ├── Content vector
       ├── Question vector
       └── Metadata representation

This can improve retrieval for complex datasets.

---

# 52. RAG with Structured Data

RAG is not limited to unstructured documents.

Knowledge can come from:

- SQL databases
- CSV
- JSON
- APIs
- Knowledge graphs

For structured data, the system may use:

    User Question
         ↓
    Query Generation
         ↓
       SQL/API
         ↓
       Results
         ↓
         LLM
         ↓
       Answer

This is related to RAG, but the retrieval mechanism is not necessarily vector similarity search.

---

# 53. RAG and SQL

For a database question:

    "What were total sales in 2025?"

A system could:

    User Question
         ↓
    LLM
         ↓
    SQL Query
         ↓
    Database
         ↓
    Query Result
         ↓
    LLM
         ↓
    Answer

This is a retrieval-augmented architecture even though vector search may not be involved.

Therefore:

> RAG is a broader architectural pattern than "vector database + LLM."

---

# 54. RAG with Web Data

A web-based RAG system can work like:

    User Query
        ↓
    Search Engine / Web Retriever
        ↓
    Web Pages
        ↓
    Extract Content
        ↓
    Relevant Context
        ↓
    LLM
        ↓
    Answer

This is useful for information that changes frequently.

---

# 55. RAG with Multiple Sources

A production system may retrieve from multiple sources:

    User Query
         ↓
    ┌───────────────┐
    │               │
    ↓               ↓
  Vector DB       SQL DB
    │               │
    ↓               ↓
  Documents      Structured Data
    │               │
    └───────┬───────┘
            ↓
        Combined Context
            ↓
           LLM
            ↓
          Answer

---

# 56. Citations in RAG

A good RAG application may return source information.

Example:

    Answer:
    Employees receive 20 annual leave days.

    Sources:
    - employee_handbook.pdf, page 5
    - leave_policy.pdf, page 2

This improves:

- Trust
- Verifiability
- Debugging
- User confidence

Metadata is especially important for citations.

---

# 57. Access Control in RAG

Security is extremely important in enterprise RAG.

Imagine:

    Employee A
       ↓
    Query
       ↓
    Retriever
       ↓
    Confidential HR document

The system must not return information the employee is not authorized to access.

Therefore retrieval should consider:

    User Identity
         +
    Permissions
         ↓
    Retrieval Filter
         ↓
    Authorized Documents

Important principle:

> Do not rely only on the LLM to enforce authorization.

Access control should be enforced at the data/retrieval layer.

---

# 58. RAG Security Risks

Important risks include:

### 1. Unauthorized Retrieval

Users retrieve documents they should not access.

### 2. Prompt Injection

Malicious content inside retrieved documents attempts to manipulate the LLM.

### 3. Data Leakage

Sensitive information appears in generated answers.

### 4. Poisoned Knowledge Base

Malicious or incorrect documents are inserted into the knowledge source.

### 5. Insecure Metadata Filters

Incorrect filters expose information across users or tenants.

### 6. Excessive Context

Sensitive irrelevant information is unnecessarily sent to the LLM.

---

# 59. RAG Evaluation

A RAG system should be evaluated separately for:

1. Retrieval quality
2. Generation quality

---

# 60. Retrieval Evaluation

Important metrics include:

### Precision

How many retrieved documents are relevant?

    Precision =
    Relevant Retrieved Documents
    /
    Total Retrieved Documents

### Recall

How many relevant documents were successfully retrieved?

    Recall =
    Relevant Retrieved Documents
    /
    Total Relevant Documents

### MRR

**Mean Reciprocal Rank**

Measures how highly the first relevant result appears.

### NDCG

**Normalized Discounted Cumulative Gain**

Measures ranking quality while considering the position of relevant results.

---

# 61. Generation Evaluation

Important aspects include:

### Faithfulness

Is the answer supported by the retrieved context?

### Answer Relevance

Does the answer actually answer the user's question?

### Correctness

Is the answer factually correct?

### Context Relevance

Is the retrieved context relevant to the question?

---

# 62. RAG Evaluation Framework

A useful evaluation structure is:

    User Query
        ↓
    Retrieval
        ↓
    Retrieved Context
        ↓
    Generated Answer

Evaluate:

    Retrieval
      ├── Precision
      ├── Recall
      ├── MRR
      └── NDCG

    Generation
      ├── Faithfulness
      ├── Relevance
      └── Correctness

---

# 63. RAG Failure Modes

## Failure 1: Wrong Chunk Retrieved

Cause:

- Poor embeddings
- Poor chunking
- Poor query

Solution:

- Improve chunking
- Improve embeddings
- Query rewriting
- Hybrid retrieval
- Reranking

---

## Failure 2: Correct Document but Wrong Context

Cause:

- Chunk too large
- Chunk too small
- Missing overlap

Solution:

- Tune chunking strategy
- Preserve document structure

---

## Failure 3: Too Many Documents

Cause:

    k too high

Problems:

- Noise
- More tokens
- Context dilution

Solution:

- Tune k
- Use reranking
- Use contextual compression

---

## Failure 4: Too Few Documents

Cause:

    k too low

Problem:

    Relevant evidence may be missing.

Solution:

- Increase k
- Improve recall
- Use multi-query retrieval

---

## Failure 5: LLM Ignores Context

Possible causes:

- Weak prompt
- Poor context formatting
- Too much irrelevant context

Solution:

- Improve prompt
- Reduce noise
- Clearly delimit context

---

## Failure 6: Hallucination

Possible causes:

- Missing evidence
- Poor retrieval
- LLM inventing unsupported information

Solution:

- Strong grounding instructions
- Better retrieval
- Citation support
- Evaluation

---

# 64. Chunking and Retrieval Relationship

Chunking has a major impact on retrieval.

Example:

    Poor Chunking
         ↓
    Poor Retrieval
         ↓
    Poor Context
         ↓
    Poor Answer

Therefore:

> RAG quality is not only an LLM problem.

It is a complete pipeline problem.

---

# 65. Embedding Model Selection

The embedding model affects retrieval quality.

Important factors:

- Semantic understanding
- Embedding dimensions
- Multilingual support
- Domain performance
- Latency
- Cost
- Maximum input size

The embedding model used during indexing must be compatible with the model used for queries.

---

# 66. Vector Dimension

Every embedding model produces vectors with a specific dimensionality.

Example:

    Document:
    [x1, x2, x3, ..., xn]

    Query:
    [y1, y2, y3, ..., yn]

For similarity comparison, the vectors need compatible dimensions.

Changing embedding models may require re-embedding the existing documents.

---

# 67. Freshness

One major advantage of RAG is that knowledge can be updated without retraining the LLM.

Example:

    Monday:
    Policy v1

    Tuesday:
    Policy v2

The knowledge base can be updated:

    New Document
         ↓
    Split
         ↓
    Embed
         ↓
    Update Vector Store

The LLM itself does not necessarily need retraining.

---

# 68. Incremental Indexing

For large systems, you do not necessarily need to rebuild everything every time.

A pipeline may detect:

    New documents
    Updated documents
    Deleted documents

and update only the affected vectors.

Conceptually:

    New/Updated Data
          ↓
      Processing
          ↓
       Embeddings
          ↓
    Vector Store Update

---

# 69. Deduplication

Duplicate documents can negatively affect retrieval.

Example:

    Document A
    Document A
    Document A

The retriever may return essentially the same information multiple times.

Solutions include:

- Content hashing
- Document IDs
- Duplicate detection
- Metadata-based deduplication

---

# 70. RAG Latency

A RAG request may involve several operations:

    Query
      ↓
    Query Embedding
      ↓
    Retrieval
      ↓
    Reranking
      ↓
    Prompt Construction
      ↓
    LLM
      ↓
    Answer

Each stage can add latency.

Production optimization may involve:

- Efficient vector indexes
- Smaller models where appropriate
- Caching
- Parallel retrieval
- Limiting retrieved documents
- Streaming generation
- Efficient reranking

---

# 71. RAG Cost

Costs can come from:

- Embedding documents
- Embedding queries
- Vector database
- Retrieval infrastructure
- Reranking
- LLM input tokens
- LLM output tokens

Reducing unnecessary context can reduce LLM token costs.

---

# 72. Caching in RAG

Caching can be used for:

- Embeddings
- Frequently used queries
- Retrieval results
- LLM responses

Example:

    Repeated Query
          ↓
       Cache Hit
          ↓
       Response

This can reduce latency and cost.

---

# 73. Production RAG Architecture

A production-grade system may look like:

    ┌──────────────────────┐
    │      Data Sources    │
    └──────────┬───────────┘
               ↓
       Ingestion Pipeline
               ↓
       Parsing / Cleaning
               ↓
          Chunking
               ↓
          Embeddings
               ↓
       Vector Database
               │
               │
    ───────────┼────────────
               │
          User Query
               ↓
       Authentication
               ↓
       Query Processing
               ↓
     Metadata / ACL Filter
               ↓
        Hybrid Retrieval
               ↓
          Reranking
               ↓
      Context Compression
               ↓
       Prompt Construction
               ↓
              LLM
               ↓
       Validation / Guardrails
               ↓
        Answer + Citations

---

# 74. RAG Components

A practical RAG system commonly contains:

| Component | Responsibility |
|---|---|
| Document Loader | Load external data |
| Text Splitter | Create chunks |
| Embedding Model | Convert text to vectors |
| Vector Store | Store/search vectors |
| Retriever | Retrieve relevant documents |
| Reranker | Improve result ordering |
| Prompt | Combine context and query |
| LLM | Generate answer |
| Output Parser | Structure output |
| Metadata | Store source information |
| Evaluation | Measure quality |
| Guardrails | Control unsafe/invalid behavior |

---

# 75. RAG vs Traditional Search

### Traditional Search

    Query
      ↓
    Search Engine
      ↓
    Documents
      ↓
    User reads documents

### RAG

    Query
      ↓
    Retriever
      ↓
    Relevant Documents
      ↓
    LLM
      ↓
    Synthesized Answer

Traditional search returns documents.

RAG uses retrieved documents to generate an answer.

---

# 76. RAG vs Chatbot

A chatbot is an application/interface.

RAG is an architecture/technique.

A chatbot can use:

    LLM only

or:

    LLM + RAG

or:

    LLM + RAG + Tools + Agents

Therefore:

> RAG is not the same thing as a chatbot.

---

# 77. RAG vs Agentic AI

### RAG

Usually follows:

    Query
      ↓
    Retrieve
      ↓
    Generate

### Agentic AI

Can involve:

    Goal
      ↓
    Reason/Plan
      ↓
    Select Tool
      ↓
    Execute
      ↓
    Observe
      ↓
    Re-plan
      ↓
    Final Answer

An agent can use RAG as one of its tools.

Example:

    Agent
      ├── Web Search
      ├── Calculator
      ├── SQL Tool
      ├── API Tool
      └── RAG Retriever

So:

> RAG can be a component inside an agentic system.

---

# 78. Knowledge Base

The collection of information used by the RAG system is often called a **knowledge base**.

It may contain:

    PDFs
    Documentation
    Database records
    FAQs
    Policies
    Product information
    Internal company documents
    Web pages

The knowledge base is the external source of information.

---

# 79. RAG Mental Model

Think of an LLM as a knowledgeable employee who does not have access to your company's filing cabinet.

RAG gives the employee a search system.

    User asks question
          ↓
    Search filing cabinet
          ↓
    Find relevant documents
          ↓
    Give documents to employee
          ↓
    Employee reads them
          ↓
    Employee answers

The LLM is the generator.

The retriever is the search assistant.

The vector database is part of the filing/search infrastructure.

---

# 80. Most Important RAG Formula

Remember:

    RAG = Retrieval + Context + Generation

More completely:

    User Query
         ↓
    Retrieve Relevant Information
         ↓
    Add Retrieved Information as Context
         ↓
    LLM Generates Answer

---

# 81. End-to-End RAG Formula

    Documents
       ↓
    Load
       ↓
    Split
       ↓
    Embed
       ↓
    Store
       ↓
    ─────────────────────
       ↓
    User Query
       ↓
    Retrieve
       ↓
    Filter / Rerank
       ↓
    Context
       ↓
    Prompt
       ↓
    LLM
       ↓
    Answer

---

# 82. Important RAG Terms

### RAG

Retrieval Augmented Generation.

### Chunk

A smaller portion of a document.

### Embedding

Numerical vector representation of semantic meaning.

### Vector Store

System used to store and search vector representations.

### Retriever

Component that retrieves relevant documents.

### Context

Information retrieved and provided to the LLM.

### Top-K

Number of results retrieved.

### Similarity Search

Search based on semantic similarity.

### MMR

Maximum Marginal Relevance.

### Reranking

Reordering retrieved candidates based on relevance.

### Hybrid Search

Combining semantic and keyword retrieval.

### Grounding

Generating an answer based on retrieved evidence.

### Hallucination

Unsupported or fabricated information generated by the model.

### Reranker

A model/component that improves ordering of retrieved candidates.

---

# 83. Common RAG Mistakes

## Mistake 1

Using extremely large chunks.

### Problem

Poor retrieval precision.

---

## Mistake 2

Using extremely small chunks.

### Problem

Important context is lost.

---

## Mistake 3

Choosing k without evaluation.

### Problem

Too many or too few documents.

---

## Mistake 4

Assuming vector search is always enough.

### Problem

Exact keywords may be missed.

### Solution

Consider hybrid search.

---

## Mistake 5

Ignoring metadata.

### Problem

Difficult filtering and citations.

---

## Mistake 6

Sending too much retrieved context.

### Problem

Higher cost and more noise.

---

## Mistake 7

Assuming RAG eliminates hallucination.

### Problem

Retrieval errors can still produce incorrect answers.

---

## Mistake 8

Ignoring document freshness.

### Problem

Old information may be retrieved.

---

## Mistake 9

Ignoring access control.

### Problem

Sensitive documents may be exposed.

---

# 84. Best Practices for RAG

1. Use appropriate document loaders.
2. Clean documents before indexing.
3. Choose chunking based on document structure.
4. Preserve useful metadata.
5. Select a suitable embedding model.
6. Use a vector database appropriate for scale.
7. Tune retrieval parameters.
8. Consider hybrid search when exact terms matter.
9. Use reranking when retrieval precision matters.
10. Avoid excessive context.
11. Add source citations.
12. Evaluate retrieval independently from generation.
13. Implement access control.
14. Keep the knowledge base fresh.
15. Monitor retrieval and generation quality.
16. Test adversarial/prompt-injection content.
17. Measure latency and cost.
18. Deduplicate documents.
19. Cache where useful.
20. Do not assume RAG automatically produces factual answers.

---

# 85. RAG Development Checklist

Before deploying a RAG system, ask:

    [ ] Are documents loaded correctly?
    [ ] Is document cleaning required?
    [ ] Is chunk size appropriate?
    [ ] Is chunk overlap appropriate?
    [ ] Are metadata fields preserved?
    [ ] Is the embedding model appropriate?
    [ ] Is the vector store appropriate?
    [ ] Is retrieval quality evaluated?
    [ ] Is k tuned?
    [ ] Is hybrid search needed?
    [ ] Is reranking needed?
    [ ] Is context compressed?
    [ ] Is the prompt grounded?
    [ ] Are citations available?
    [ ] Is access control enforced?
    [ ] Is stale data handled?
    [ ] Are hallucinations evaluated?
    [ ] Are latency and cost measured?
    [ ] Are security threats tested?

---

# 86. Interview Questions

## Beginner

### Q1. What is RAG?

RAG is an architecture that retrieves relevant external information and provides it to an LLM as context before generating an answer.

### Q2. What does RAG stand for?

Retrieval Augmented Generation.

### Q3. Why is RAG used?

To provide external, private, current, or domain-specific information to an LLM without necessarily retraining the model.

### Q4. What are the main steps of RAG?

    Load → Split → Embed → Store → Retrieve → Augment → Generate

### Q5. What is a retriever?

A component that accepts a query and returns relevant documents.

### Q6. What is an embedding?

A numerical vector representation of text or another piece of data that captures semantic information.

### Q7. What is a vector store?

A system that stores vector representations and supports similarity-based retrieval.

---

# 87. Intermediate Interview Questions

### Q8. What is the difference between a vector store and a retriever?

A vector store stores and searches vectors, while a retriever provides an interface for retrieving relevant documents.

### Q9. What is chunking?

Dividing large documents into smaller pieces suitable for retrieval and LLM processing.

### Q10. Why is chunk overlap used?

To preserve context across chunk boundaries.

### Q11. What is MMR?

Maximum Marginal Relevance balances relevance with diversity when selecting retrieved documents.

### Q12. What is hybrid search?

Combining semantic/vector retrieval with keyword-based retrieval.

### Q13. What is reranking?

Taking retrieved candidates and reordering them using a stronger relevance model.

### Q14. Does RAG eliminate hallucination?

No. RAG can reduce unsupported answers when retrieval and prompting are good, but hallucinations can still occur.

### Q15. RAG vs fine-tuning?

RAG primarily supplies external knowledge at inference time, while fine-tuning changes model behavior through additional training.

---

# 88. Advanced Interview Questions

### Q16. How would you improve a RAG system with poor retrieval?

Investigate:

    Chunking
    Embeddings
    Query formulation
    Metadata filtering
    k
    Hybrid retrieval
    Multi-query retrieval
    Reranking

### Q17. How do you evaluate RAG?

Separate evaluation into:

    Retrieval:
    Precision
    Recall
    MRR
    NDCG

    Generation:
    Faithfulness
    Relevance
    Correctness

### Q18. Why might semantic search fail?

Semantic search can struggle with exact identifiers, rare terms, codes, names, or cases where lexical matching is important.

Use hybrid retrieval when appropriate.

### Q19. How can you reduce RAG latency?

Possible techniques:

    Caching
    Efficient indexes
    Smaller/faster embedding models
    Parallel retrieval
    Limiting retrieved results
    Efficient reranking
    Streaming LLM output

### Q20. How can you secure an enterprise RAG system?

Use:

    Authentication
    Authorization
    Metadata/ACL filtering
    Tenant isolation
    Secure ingestion
    Prompt-injection defenses
    Data leakage controls
    Logging and auditing

---

# 89. One-Line Revision

    RAG retrieves relevant external knowledge,
    adds it to the LLM context,
    and uses the LLM to generate a grounded answer.

---

# 90. Quick Revision Flow

    ┌──────────────────┐
    │   Data Sources   │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │ Document Loader  │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │  Text Splitter   │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │   Embeddings     │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │   Vector Store   │
    └────────┬─────────┘
             │
             │
         User Query
             ↓
    ┌──────────────────┐
    │    Retriever     │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │ Relevant Context │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │      Prompt      │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │       LLM        │
    └────────┬─────────┘
             ↓
    ┌──────────────────┐
    │   Final Answer   │
    └──────────────────┘

---

# 91. Final Mental Model

Remember RAG using these 7 words:

    LOAD
      ↓
    SPLIT
      ↓
    EMBED
      ↓
    STORE
      ↓
    RETRIEVE
      ↓
    AUGMENT
      ↓
    GENERATE

### In simple language:

    Load the knowledge
          ↓
    Break it into chunks
          ↓
    Convert chunks into vectors
          ↓
    Store the vectors
          ↓
    Find relevant chunks
          ↓
    Give those chunks to the LLM
          ↓
    Generate the answer

### Ultimate Formula

    RAG

    = External Knowledge
    + Retrieval
    + Context
    + LLM Generation

### Most important concept:

    RAG does NOT teach the LLM new knowledge permanently.

    RAG retrieves knowledge at query time
    and temporarily provides that knowledge
    as context to the LLM.

That is the core idea behind Retrieval Augmented Generation.